# 03_02 — LLM Fine-tuning (Unsloth)

Fine-tune a small instruction LLM on density-weighted hate speech data using Unsloth + LoRA.

**Design**
- Training samples are drawn from the same `densities.csv` produced in notebook 02.
- `K` and `SPACE` control which density column is used for weighting — pick the values that
  performed best in the general encoder runs (notebook 04, section 3).
- The fine-tuned model is saved to `outputs/3_training/llm_finetuned/` and evaluated
  in the companion notebook **04_01_evaluation_llm**.

**Sections**
1. Configuration
2. Select density parameters (K + space)
3. Fine-tune
4. Training summary

In [ ]:
import sys, os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import json
import yaml
import pandas as pd

from src.training import TrainingConfig
from src.training.llm import train_llm, PROMPT_TEMPLATES

## 1. Configuration

In [ ]:
CONFIG_PATH = 'configs/datasets.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

EMBEDDINGS_ROOT   = cfg['embedding']['output_root']
PREPROCESSED_ROOT = cfg['preprocessing']['output_root']
TRAINING_ROOT     = cfg['training']['output_root']
MODEL_SLUG        = cfg['embedding']['models'][0].replace('/', '_')
K_VALUES          = cfg['embedding']['k_values']

llm_cfg = cfg['llm_finetuning']

config = TrainingConfig(
    output_root      = llm_cfg['output_root'],
    unsloth_model    = llm_cfg['unsloth_model'],
    lora_r           = llm_cfg['lora_r'],
    lora_alpha       = llm_cfg['lora_alpha'],
    llm_train_size   = llm_cfg['train_size'],
    llm_max_steps    = llm_cfg['max_steps'],
    llm_batch_size   = llm_cfg['batch_size'],
    llm_learning_rate= llm_cfg['learning_rate'],
    random_state     = cfg['preprocessing']['random_state'],
)

print('Unsloth model  :', config.unsloth_model)
print('LoRA r / alpha :', config.lora_r, '/', config.lora_alpha)
print('Train size     :', config.llm_train_size)
print('Max steps      :', config.llm_max_steps)
print('K values       :', K_VALUES)
print('Prompt modes   :', list(PROMPT_TEMPLATES.keys()))

## 2. Select Density Parameters

Choose `K` and `SPACE` based on the best encoder model configuration from **notebook 04**.

| Variable | Options | Meaning |
|----------|---------|----------|
| `K`      | any value in `K_VALUES` | neighbourhood size for density estimation |
| `SPACE`  | `'raw'` or `'pca'`      | embedding space used for density |

These control which column from `densities.csv` is used to weight training samples.

In [ ]:
# --- Set these based on the best general model in notebook 04 ---
K     = 100    # best K value
SPACE = 'pca'  # 'raw' or 'pca'

# Dataset to train on (Russian is excluded — it is the evaluation target)
DATASET = 'toxigen'

density_col = f"density{'_pca' if SPACE == 'pca' else ''}_k{K}_ratio"

DENSITY_CSV  = os.path.join(EMBEDDINGS_ROOT, DATASET, MODEL_SLUG, 'densities.csv')
TEST_CSV     = os.path.join(PREPROCESSED_ROOT, 'russian', 'test.csv')

print(f'Density column : {density_col}')
print(f'Density CSV    : {DENSITY_CSV}  exists={os.path.exists(DENSITY_CSV)}')
print(f'Test CSV       : {TEST_CSV}  exists={os.path.exists(TEST_CSV)}')

# Preview density distribution
if os.path.exists(DENSITY_CSV):
    df_preview = pd.read_csv(DENSITY_CSV)
    print(f'\nDensities CSV shape: {df_preview.shape}')
    density_cols = [c for c in df_preview.columns if c.startswith('density')]
    print(f'Available density columns:\n  ' + '\n  '.join(density_cols))
    if density_col in df_preview.columns:
        print(f'\n{density_col} stats:')
        print(df_preview[density_col].describe())
    else:
        print(f'\n⚠ Column "{density_col}" not found — adjust K or SPACE above.')

## 3. Fine-tune

Set `RUN_TRAINING = True` the first time.  
Subsequent runs can load the cached `metrics.json` without re-training.

> **Requirements**: Unsloth, TRL, and a CUDA-capable GPU.  
> Install: `pip install unsloth trl datasets`

In [ ]:
RUN_TRAINING = False  # set True to fine-tune (requires GPU + unsloth)

model_slug   = config.unsloth_model.replace('/', '_').replace(':', '_')
density_tag  = f"{'pca_' if SPACE == 'pca' else ''}k{K}_ratio"
run_out_dir  = os.path.join(
    config.output_root, 'llm_finetuned', model_slug,
    f'{DATASET}__{density_tag}',
)
metrics_path = os.path.join(run_out_dir, 'metrics.json')

if RUN_TRAINING:
    metrics = train_llm(
        density_csv = DENSITY_CSV,
        test_csv    = TEST_CSV,
        dataset_tag = DATASET,
        k           = K,
        space       = SPACE,
        config      = config,
    )
    print('Training complete.')
elif os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(f'Loaded cached metrics from {metrics_path}')
else:
    metrics = None
    print(f'No cached metrics found at {metrics_path}. Set RUN_TRAINING=True to train.')

## 4. Training Summary

In [ ]:
if metrics:
    print(f"Model        : {config.unsloth_model}")
    print(f"Dataset      : {DATASET}  |  K={K}  |  space={SPACE}")
    print(f"Train size   : {config.llm_train_size}  |  max_steps={config.llm_max_steps}")
    print()
    for key in ['f1', 'balanced_accuracy', 'accuracy', 'precision', 'recall', 'training_loss']:
        val = metrics.get(key)
        if val is not None:
            print(f"  {key:<22}: {val:.4f}")
    print()
    print(f"Model saved  : {os.path.join(run_out_dir, 'model')}")
    print("→ Run 04_01_evaluation_llm.ipynb for full evaluation and comparison plots.")
else:
    print('No metrics to display.')